In [2]:
import pandas as pd
import scanpy as sc
import anndata as ad
from torch_geometric.graphgym import train

In [3]:
# 载入单细胞扰动数据
rep_ad = sc.read_h5ad('E:\\rpe1_raw_singlecell_01.h5ad')

In [4]:
# get cross gene
ad_test = sc.read_h5ad("predictions/aligned_hepg2_test.h5ad")

cross_gene = ad_test.var_names.intersection(rep_ad.var['gene_name'])

len(cross_gene)

6207

In [5]:
import numpy as np

# 随机选择30%的基因作为验证基因
np.random.seed(42)  # 设置随机种子以保证可重复性
validation_size = int(len(cross_gene) * 0.3)
validation_genes = np.random.choice(cross_gene, size=validation_size, replace=False)

train_genes = np.setdiff1d(cross_gene, validation_genes)
print(len(train_genes))
print(len(validation_genes))
print(len(cross_gene))

4345
1862
6207


In [6]:
rep_ad_all = rep_ad[:,rep_ad.var['gene_name'].isin(cross_gene)]
rep_ad_all.shape

(247914, 6207)

In [7]:
pert_all = pd.read_csv("data/node_map_pert.csv",index_col=0).index
print(len(pert_all))
print(pert_all[:5])

9853
Index(['A1BG', 'AAAS', 'AACS', 'AAGAB', 'AAK1'], dtype='object')


In [8]:
pert_in_rep = rep_ad_all.obs.gene.unique()
print(len(pert_in_rep))

2394


In [9]:
inter_pert = np.intersect1d(pert_all, pert_in_rep)
print(len(inter_pert))

2373


In [10]:
# 随机选择20%的扰动作为验证扰动
np.random.seed(42)  # 设置随机种子以保证可重复性
validation_size = int(len(inter_pert) * 0.2)
validation_perts = np.random.choice(inter_pert, size=validation_size, replace=False)
train_perts = np.setdiff1d(inter_pert, validation_perts)

print(len(train_perts))
print(len(validation_perts))
print(len(inter_pert))

1899
474
2373


In [11]:
# 添加condition列
print("\nAdding condition column...")
# 对于non-targeting设为ctrl，其他设为gene+ctrl
rep_ad_all.obs['condition'] = rep_ad_all.obs['gene'].apply(lambda x: 'ctrl' if x == 'non-targeting' else f'{x}+ctrl')
print("\nCondition column (first 5 rows):")
print(rep_ad_all.obs['condition'].head())
print("\nScript completed successfully!")


Adding condition column...


C:\Users\kisun\AppData\Local\Temp\ipykernel_22568\692556260.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  rep_ad_all.obs['condition'] = rep_ad_all.obs['gene'].apply(lambda x: 'ctrl' if x == 'non-targeting' else f'{x}+ctrl')



Condition column (first 5 rows):
cell_barcode
AAACCCAAGAAACTAC-53      MRPS31+ctrl
AAACCCAAGAAGCCAC-51    LRRC37A3+ctrl
AAACCCAAGAAGCGAA-32       SRCAP+ctrl
AAACCCAAGAATACAC-44        WBP1+ctrl
AAACCCAAGAATCGAT-43       RRP12+ctrl
Name: condition, dtype: category
Categories (2394, object): ['AAAS+ctrl', 'AAMP+ctrl', 'AAR2+ctrl', 'AARS+ctrl', ..., 'ZRSR2+ctrl', 'ZW10+ctrl', 'ZWINT+ctrl', 'ctrl']

Script completed successfully!


In [12]:
rep_ad_all.obs['cell_type'] = 'rpe1'

In [13]:
train_rep = rep_ad_all[rep_ad_all.obs['gene'].isin(train_perts) | (rep_ad_all.obs['gene'] == 'non-targeting')]
test_rep = rep_ad_all[rep_ad_all.obs['gene'].isin(validation_perts) | (rep_ad_all.obs['gene'] == 'non-targeting')]

In [14]:
print(train_rep.shape)
print(test_rep.shape)

(193171, 6207)
(63631, 6207)


In [15]:
train_rep.obs['condition'].value_counts()

condition
ctrl           11485
MRPL36+ctrl     1686
TARDBP+ctrl     1675
PPP6C+ctrl      1581
NBPF12+ctrl     1159
               ...  
INCENP+ctrl        4
ZC3H8+ctrl         4
CDK7+ctrl          4
NAT10+ctrl         3
NPAT+ctrl          3
Name: count, Length: 1900, dtype: int64

In [16]:
test_rep.obs['condition'].value_counts()

condition
ctrl            11485
TFAM+ctrl        3580
SLC1A5+ctrl      1962
GFM1+ctrl        1699
MRPL35+ctrl      1491
                ...  
RPF2+ctrl           8
PIP4K2C+ctrl        7
TOP2A+ctrl          6
RPS16+ctrl          5
NUP93+ctrl          2
Name: count, Length: 475, dtype: int64

In [17]:
train_rep_mask = train_rep[:,train_rep.var['gene_name'].isin(train_genes)]
test_rep_mask = test_rep[:,test_rep.var['gene_name'].isin(train_genes)]
test_rep_mask_true = test_rep[:,test_rep.var['gene_name'].isin(validation_genes)]

print(train_rep_mask.shape)
print(test_rep_mask.shape)
print(test_rep_mask_true.shape)

(193171, 4345)
(63631, 4345)
(63631, 1862)


In [18]:
train_rep.write_h5ad('data/rpe1/rep_rpe1_train.h5ad')
test_rep.write_h5ad('data/rpe1/rep_rpe1_test.h5ad')

train_rep_mask.write_h5ad('data/rpe1/rep_rpe1_train_mask.h5ad')
test_rep_mask.write_h5ad('data/rpe1/rep_rpe1_test_mask.h5ad')
test_rep_mask_true.write_h5ad('data/rpe1/rep_rpe1_test_mask_true.h5ad')


D:\miniconda3\envs\gears_stack\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
D:\miniconda3\envs\gears_stack\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
D:\miniconda3\envs\gears_stack\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
D:\miniconda3\envs\gears_stack\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
D:\miniconda3\envs\gears_stack\lib\site-packages\anndata\_core\anndata.py:1138: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
